# 0.8+ Audio Deepfake Detection Pipeline

GPU T4 연결 후 → **런타임 → 모두 실행** 하면 끝.

In [4]:
# 셀 0: GPU 확인
!nvidia-smi

Wed Aug 26 17:36:34 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   41C    P8             13W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [5]:
# 셀 1: 환경 세팅
!pip -q install librosa soundfile transformers accelerate demucs panns-inference onnxruntime-gpu datasets huggingface_hub scikit-learn scipy edge-tts
!apt -qq install -y ffmpeg > /dev/null
print('Done')



Done


In [6]:
# 셀 2: Drive 마운트
from google.colab import drive
drive.mount('/content/drive', force_remount=True)
print('Drive mounted!')

Mounted at /content/drive
Drive mounted!


In [7]:
# 셀 3: 프로젝트 경로로 이동
import os

# Google Drive에서 dacon1 폴더 찾기
drive_path = '/content/drive/MyDrive'
project_name = 'dacon1'

# 대소문자 무시하고 검색
found = False
for item in os.listdir(drive_path):
    if item.lower() == project_name.lower():
        full_path = os.path.join(drive_path, item)
        if os.path.isdir(full_path):
            os.chdir(full_path)
            print(f'프로젝트 폴더 발견: {full_path}')
            found = True
            break

if not found:
    print('ERROR: dacon1 폴더를 찾을 수 없습니다.')
    print(f'\nGoogle Drive 폴더 목록:')
    for item in sorted(os.listdir(drive_path)):
        print(f'  - {item}')
    print('\n\n>> dacon1 폴더를 만들고 파일을 업로드하세요!')
else:
    print(f'\n현재 디렉토리: {os.getcwd()}')
    print('\n폴더 내용:')
    !ls -la

ERROR: dacon1 폴더를 찾을 수 없습니다.

Google Drive 폴더 목록:
  - 1710559668251.jpg
  - 2.mov
  - 2017-11-18_22'54'49_174.mp3
  - 20190814_220419.jpg
  - 20190821_215139.mp4
  - 20190826_163551.jpg
  - 20190826_163601.mp4
  - 20190828_162129.jpg
  - 20190828_162159.jpg
  - 20190828_174936.jpg
  - 20190828_174938.jpg
  - 20190901_012128.jpg
  - 20190916_122313.jpg
  - 20190922_002325.jpg
  - 20190922_151305.jpg
  - 20190926_165635.jpg
  - 20190926_165644.jpg
  - 20191002_145902.mp4
  - 20191007_132529.mp4
  - 20191008_123158.jpg
  - 20191024_202329.jpg
  - 20191029_114721.jpg
  - 20191029_114747.jpg
  - 20191029_114756.jpg
  - 20191029_114807.jpg
  - 20191029_114829.jpg
  - 20191029_114842.jpg
  - 20191029_114848.jpg
  - 20191029_114854.jpg
  - 20191029_114907.jpg
  - 20191030_145523.jpg
  - 20191030_145531.jpg
  - 20191030_145545.jpg
  - 20191030_212252.jpg
  - 20191104_172620.jpg
  - 20191128_170053.jpg
  - 20191204_151250.jpg
  - 20191204_151252.jpg
  - 20191204_152119.jpg
  - 20191208_101421.jp

In [8]:
# 셀 4: 필수 파일 확인
import os

required = [
    'scripts/build_train_data.py',
    'scripts/train_raptor.py',
    'submit/script_v2.py',
    'data/sample_submission.csv'
]

all_ok = True
for f in required:
    if os.path.exists(f):
        print(f'  OK: {f}')
    else:
        print(f'  MISSING: {f}')
        all_ok = False

if all_ok:
    print('\n✅ 모든 필수 파일 준비 완료!')
else:
    print('\n❌ 누락된 파일이 있습니다. 업로드하세요!')

  MISSING: scripts/build_train_data.py
  MISSING: scripts/train_raptor.py
  MISSING: submit/script_v2.py
  MISSING: data/sample_submission.csv

❌ 누락된 파일이 있습니다. 업로드하세요!


In [9]:
# 셀 5: 학습 데이터 구축 (~30분)
!python scripts/build_train_data.py \
    --out train_data \
    --auto-download \
    --max-voice-real 2000 \
    --max-voice-fake 3000 \
    --max-music-real 500 \
    --max-music-fake 2000

python3: can't open file '/content/scripts/build_train_data.py': [Errno 2] No such file or directory


In [10]:
# 셀 6: RAPTOR 학습 (~4시간)
!python scripts/train_raptor.py \
    --train train_data/manifest_train.csv \
    --val train_data/manifest_val.csv \
    --backbone utter-project/mHuBERT-147 \
    --out runs/raptor_v1 \
    --epochs 20 --bs 24 --lr 1e-6 \
    --lr-head 3e-4 --consistency-w 0.25 --p-aug 0.5

python3: can't open file '/content/scripts/train_raptor.py': [Errno 2] No such file or directory


In [11]:
# 셀 7: 학습 결과 확인
import pandas as pd
import os

log_path = 'runs/raptor_v1/log.csv'
if os.path.exists(log_path):
    log = pd.read_csv(log_path)
    print(log.to_string(index=False))
    print(f'\nBest val EER: {log["val_eer"].min():.4f}')
else:
    print(f'ERROR: {log_path} not found!')
    print('runs/raptor_v1/ 폴더 내용:')
    !ls -la runs/

ERROR: runs/raptor_v1/log.csv not found!
runs/raptor_v1/ 폴더 내용:
ls: cannot access 'runs/': No such file or directory


In [12]:
# 셀 8: 모델 복사 + 제출 추론
!cp runs/raptor_v1/best.pth submit/model/raptor_best.pth
!python submit/script_v2.py \
    --test-dir data/test \
    --sample-submission data/sample_submission.csv \
    --output output/submission.csv \
    --device cuda --tta 2

cp: cannot stat 'runs/raptor_v1/best.pth': No such file or directory
python3: can't open file '/content/submit/script_v2.py': [Errno 2] No such file or directory


In [13]:
# 셀 9: 제출 zip 생성
!python scripts/build_submit_zip.py \
    --script submit/script_v2.py \
    --output submit.zip

python3: can't open file '/content/scripts/build_submit_zip.py': [Errno 2] No such file or directory


In [14]:
# 셀 10: 결과 다운로드
from google.colab import files
files.download('submit.zip')
files.download('output/submission.csv')

FileNotFoundError: Cannot find file: submit.zip